# Dijet data run-ID dependence

Compare reconstructed dijet CM-frame pseudorapidity shapes for the five diagnostic run IDs with the run-inclusive control. Each projection is independently normalized to unit integral. Run overlays and run/inclusive ratios are drawn on separate canvases for every interval in `TEST_DIJET_PTAVE_BINS`.

<!-- detailed-workflow-guide -->

### Detailed workflow and stability test

Run-dependent projections test whether a selected observable is stable across acquisition periods. For run group $r$ and observable bin $i$, a normalized comparison may use $p_{ir}=N_{ir}/\sum_jN_{jr}$; an unnormalized comparison retains changing exposure. Ratios must therefore be interpreted according to the configured normalization.

A trend can reflect luminosity, trigger prescales, detector conditions, or physics composition. This notebook diagnoses stored distributions but does not derive a calibration correction. Run boundaries and excluded periods are analysis inputs and should be changed only with an explicit data-quality justification.

## Environment and imports

Start Jupyter from the repository root with `py-env/bin/python -m jupyter notebook`.

In [ ]:
# Cell role: initialize the reproducible Python/ROOT environment and shared helpers.
# Interpretation: No physics histogram is modified here; ROOT ownership is configured before files open.
# The preceding Markdown gives the equations and physics assumptions for this step.
%load_ext autoreload
%autoreload 2

from dataclasses import replace
from pathlib import Path
import os
import sys

PROJECT_ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / 'CMakeLists.txt').is_file()
     and (candidate / 'hist_analysis').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('Cannot locate jetAnalysis; start Jupyter from its root.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hist_analysis.python.notebook_setup import load_root
ROOT = load_root(batch=True)

from hist_analysis.config.files import BASE_DIR
from hist_analysis.config.histograms import COMMON_ETA_CM_RANGE, TEST_DIJET_PTAVE_BINS
from hist_analysis.python.data_run_dependence import draw_run_dependence
from hist_analysis.python.histogram_io import resolve_data_file
from hist_analysis.python.root_style import DEFAULT_PLOT_STYLE

In [ ]:
# Cell role: perform analysis step 2.
# Interpretation: Operations use the binning, normalization, and uncertainty conventions documented above.
# The preceding Markdown gives the equations and physics assumptions for this step.
ROOT.gStyle.SetOptStat(0)
ROOT.gStyle.SetPalette(ROOT.kBird)
ROOT.TH1.AddDirectory(False)

## Configuration

`RATIO_OPTION` controls only the run/inclusive uncertainty propagation: use `''` for ROOT's standard independent errors or `'B'` for binomial errors. The stored diagnostic histograms all use the fixed selection `|#eta_{CM}^{jet}| < 1.9`.

In [ ]:
# Cell role: define and validate user-facing analysis configuration.
# Interpretation: Changing these values can change inputs, selections, binning, normalization, or outputs.
# The preceding Markdown gives the equations and physics assumptions for this step.
DATA_DIR = Path(os.environ.get('PPB_DATA_DIR', BASE_DIR / 'exp'))
DATA_DIRECTION = 'combined'  # combined, Pbgoing, or pgoing
DATA_SELECTION = 'jetId'     # jetId, trkMax, or noSel
TRIGGERS = ('MinimumBias', 'Jet60', 'Jet80', 'Jet100')
DATA_FILES = {
    trigger: resolve_data_file(DATA_DIR, trigger, DATA_DIRECTION, DATA_SELECTION)
    for trigger in TRIGGERS
}
OUTPUT_DIR = Path(os.environ.get(
    'DATA_RUNID_OUTPUT_DIR',
    PROJECT_ROOT / 'hist_analysis/output/data_runId_dependence',
)) / DATA_DIRECTION / DATA_SELECTION
REBIN_ETA = 2
ETA_DISPLAY_RANGE = COMMON_ETA_CM_RANGE
RATIO_RANGE = (0.75, 1.25)
RATIO_OPTION = 'B'  # '' for independent errors, 'B' for binomial errors
SAVE_PNG = False
DRAW_GRID = True
PLOT_STYLE = replace(
    DEFAULT_PLOT_STYLE, annotation_text_size=0.026, legend_text_size=0.026,
)

if DATA_DIRECTION not in {'combined', 'Pbgoing', 'pgoing'}:
    raise ValueError(f'Unsupported DATA_DIRECTION={DATA_DIRECTION!r}')
if DATA_SELECTION not in {'jetId', 'trkMax', 'noSel'}:
    raise ValueError(f'Unsupported DATA_SELECTION={DATA_SELECTION!r}')
if isinstance(REBIN_ETA, bool) or not isinstance(REBIN_ETA, int) or REBIN_ETA < 1:
    raise ValueError('REBIN_ETA must be a positive integer')
if RATIO_OPTION not in {'', 'B'}:
    raise ValueError("RATIO_OPTION must be '' or 'B'")
missing = [str(path) for path in DATA_FILES.values() if not path.exists()]
if missing:
    raise FileNotFoundError('Missing ROOT files:\n' + '\n'.join(missing))
DATA_FILES

## MinimumBias

In [ ]:
# Cell role: construct derived ratios, efficiencies, or correction factors.
# Interpretation: The numerator/denominator relationship determines whether independent or binomial errors are valid.
# The preceding Markdown gives the equations and physics assumptions for this step.
mb_results = draw_run_dependence(
    'MinimumBias', DATA_FILES['MinimumBias'], TEST_DIJET_PTAVE_BINS,
    output_dir=OUTPUT_DIR, rebin_eta=REBIN_ETA, eta_range=ETA_DISPLAY_RANGE,
    ratio_range=RATIO_RANGE, ratio_option=RATIO_OPTION,
    save_png=SAVE_PNG, grid=DRAW_GRID, style=PLOT_STYLE,
)

## Jet60

In [ ]:
# Cell role: construct derived ratios, efficiencies, or correction factors.
# Interpretation: The numerator/denominator relationship determines whether independent or binomial errors are valid.
# The preceding Markdown gives the equations and physics assumptions for this step.
jet60_results = draw_run_dependence(
    'Jet60', DATA_FILES['Jet60'], TEST_DIJET_PTAVE_BINS,
    output_dir=OUTPUT_DIR, rebin_eta=REBIN_ETA, eta_range=ETA_DISPLAY_RANGE,
    ratio_range=RATIO_RANGE, ratio_option=RATIO_OPTION,
    save_png=SAVE_PNG, grid=DRAW_GRID, style=PLOT_STYLE,
)

## Jet80

In [ ]:
# Cell role: construct derived ratios, efficiencies, or correction factors.
# Interpretation: The numerator/denominator relationship determines whether independent or binomial errors are valid.
# The preceding Markdown gives the equations and physics assumptions for this step.
jet80_results = draw_run_dependence(
    'Jet80', DATA_FILES['Jet80'], TEST_DIJET_PTAVE_BINS,
    output_dir=OUTPUT_DIR, rebin_eta=REBIN_ETA, eta_range=ETA_DISPLAY_RANGE,
    ratio_range=RATIO_RANGE, ratio_option=RATIO_OPTION,
    save_png=SAVE_PNG, grid=DRAW_GRID, style=PLOT_STYLE,
)

## Jet100

In [ ]:
# Cell role: construct derived ratios, efficiencies, or correction factors.
# Interpretation: The numerator/denominator relationship determines whether independent or binomial errors are valid.
# The preceding Markdown gives the equations and physics assumptions for this step.
jet100_results = draw_run_dependence(
    'Jet100', DATA_FILES['Jet100'], TEST_DIJET_PTAVE_BINS,
    output_dir=OUTPUT_DIR, rebin_eta=REBIN_ETA, eta_range=ETA_DISPLAY_RANGE,
    ratio_range=RATIO_RANGE, ratio_option=RATIO_OPTION,
    save_png=SAVE_PNG, grid=DRAW_GRID, style=PLOT_STYLE,
)